In [1]:
import json
import re
import time
import logging
import requests
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

logging.basicConfig(level=logging.WARNING)

In [2]:
# Read in dataset — sourced from a Kaggle dataset on Lookfantastic.com skincare products
sc_data = pd.read_csv('skincare_products_2021.csv')
print(f"Loaded {len(sc_data)} products | Columns: {sc_data.columns.tolist()}")
sc_data.head(2)

Loaded 1138 products | Columns: ['product_name', 'product_url', 'product_type', 'clean_ingreds', 'price']


,product_name,product_url,product_type,clean_ingreds,price
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",£5.20
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",£13.00


In [3]:
# Rename ingredients column and strip currency symbol from price
sc_data = sc_data.rename(columns={'clean_ingreds': 'ingredients'})
sc_data['price'] = sc_data['price'].str.replace('£', '', regex=False).astype(float)
sc_data.head(2)

,product_name,product_url,product_type,ingredients,price
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",5.2
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",13.0


In [4]:
# --- Scraping helpers ---

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-GB,en;q=0.9',
}


def extract_json_ld_product(soup):
    """Return the first Product JSON-LD object found on a page."""
    for script in soup.find_all('script', type='application/ld+json'):
        try:
            data = json.loads(script.string)
            items = data.get('@graph', [data]) if isinstance(data, dict) else [data]
            for item in items:
                if isinstance(item, dict) and item.get('@type') in ('Product', ['Product']):
                    return item
        except (json.JSONDecodeError, TypeError, AttributeError):
            continue
    return None


def scrape_product(session, url, idx):
    """Fetch current price, rating, image URL, and brand for one product page."""
    result = {
        'idx': idx,
        'updated_price': None,
        'product_rating': None,
        'product_image_url': None,
        'brand_from_page': None,
        'available': True,
    }
    try:
        resp = session.get(url, headers=HEADERS, timeout=15)
        if resp.status_code == 404:
            result['available'] = False
            return result
        resp.raise_for_status()

        soup = BeautifulSoup(resp.content, 'html.parser')
        product = extract_json_ld_product(soup)

        if product:
            # Price
            offers = product.get('offers', {})
            if isinstance(offers, list):
                offers = offers[0] if offers else {}
            if isinstance(offers, dict):
                try:
                    result['updated_price'] = float(offers.get('price', 0) or 0) or None
                except (ValueError, TypeError):
                    pass

            # Rating
            agg = product.get('aggregateRating', {})
            if agg:
                try:
                    result['product_rating'] = float(agg.get('ratingValue', 0))
                except (ValueError, TypeError):
                    pass

            # Image
            image = product.get('image')
            if isinstance(image, list) and image:
                img = image[0]
                result['product_image_url'] = img.get('url') if isinstance(img, dict) else img
            elif isinstance(image, dict):
                result['product_image_url'] = image.get('url')
            elif isinstance(image, str):
                result['product_image_url'] = image

            # Brand
            brand_data = product.get('brand', {})
            if isinstance(brand_data, dict):
                result['brand_from_page'] = brand_data.get('name')
            elif isinstance(brand_data, str):
                result['brand_from_page'] = brand_data

    except requests.exceptions.HTTPError:
        result['available'] = False
    except Exception:
        pass

    return result

In [5]:
# --- Concurrent scraping with progress tracking ---

MAX_WORKERS = 15       # parallel threads
REQUEST_DELAY = 0.05   # polite per-request delay (seconds)

session = requests.Session()
urls = list(enumerate(sc_data['product_url']))


def _scrape(args):
    idx, url = args
    time.sleep(REQUEST_DELAY)
    return scrape_product(session, url, idx)


results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(_scrape, item): item[0] for item in urls}
    for future in tqdm(as_completed(futures), total=len(urls), desc='Scraping products'):
        try:
            results.append(future.result())
        except Exception as exc:
            logging.warning('Scrape error: %s', exc)

# Apply results back to dataframe
for r in results:
    i = r['idx']
    sc_data.at[i, 'updated_price']    = r['updated_price']
    sc_data.at[i, 'product_rating']   = r['product_rating']
    sc_data.at[i, 'product_image_url']= r['product_image_url']
    sc_data.at[i, 'available']        = r['available']
    sc_data.at[i, 'brand_from_page']  = r['brand_from_page']

sc_data['updated_price'] = pd.to_numeric(sc_data['updated_price'], errors='coerce')
sc_data['price_change']  = sc_data['updated_price'] - sc_data['price']

sc_data['available'] = sc_data['available'].astype(bool)
available    = sc_data['available'].sum()
discontinued = (~sc_data['available']).sum()
print(f"Available: {available}  |  Discontinued / unreachable: {discontinued}")

Scraping products:   0%|          | 0/1138 [00:00<?, ?it/s]

Available: 1032  |  Discontinued / unreachable: -2170


In [ ]:
# Scrape brand list from Lookfantastic 
# The site was redesigned; we try two selectors and fall back to a cached list.

def fetch_brand_list(url='https://www.lookfantastic.com/brands.list'):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')

        # New site structure: brand names in anchor tags within brand listing sections
        brand_anchors = soup.select('a[href*=".list"]')
        names = [
            a.get_text(strip=True)
            for a in brand_anchors
            if a.get_text(strip=True) and len(a.get_text(strip=True)) > 1
               and not a['href'].startswith('/c/')  # skip category links
        ]

        # Legacy selector fallback
        if len(names) < 50:
            panels = soup.find_all('div', class_='responsiveBrandsPageScroll_panel')
            raw = [p.text.strip().replace('\n', ' ') for p in panels]
            names = [b for entry in raw for b in filter(None, entry.split('   '))]

        return sorted(set(names), key=len, reverse=True) if names else None

    except Exception as exc:
        logging.warning('Brand list fetch failed: %s', exc)
        return None


# Cached brand list from last successful scrape (2024)
CACHED_BRAND_LIST = [
    'Kylie Cosmetics by Kylie Jenner', 'Obsessive Compulsive Cosmetics',
    'Jean Paul Gaultier Fragrance', 'The Vintage Cosmetic Company',
    'Lab Series Skincare for Men', 'Anthony Logistics for Men',
    'Experimental Perfume Club', 'Shiseido Skincare For Men',
    'Anastasia Beverly Hills', 'Aromatherapy Associates',
    'NYX Professional Makeup', 'Sebastian Professional',
    'Shu Uemura Art of Hair', 'Youth To The People',
    'REN Clean Skincare', 'Revolution Beauty', 'Augustinus Bader',
    'Beauty of Joseon', 'Carolina Herrera', 'Christophe Robin',
    'Dr. Dennis Gross', 'First Aid Beauty', 'Jo Malone London',
    'Josh Wood Colour', 'MAKE UP FOR EVER', 'Pat McGrath Labs',
    'Umberto Giannini', 'Atelier Cologne', 'Dolce & Gabbana',
    'Elizabeth Arden', 'Grown Alchemist', 'Kate Somerville',
    'La Roche-Posay', 'Natasha Denona', 'NEOM Wellbeing',
    'Sanctuary Spa', 'Sol de Janeiro', 'The INKEY List',
    'Tommy Hilfiger', 'Verso Skincare', 'American Crew',
    'BLEACH LONDON', 'Floral Street', 'Floris London',
    'Grow Gorgeous', 'Holika Holika', 'Honest Beauty',
    'ICONIC London', 'Jessica Nails', 'Kora Organics',
    "L'Oréal Paris", 'Laura Mercier', 'Lottie London',
    'MAC Cosmetics', 'Manuka Health', 'Mario Badescu',
    'Ole Henriksen', 'Paul Mitchell', 'Sarah Chapman',
    'Shea Moisture', 'SkinCeuticals', 'Tangle Teezer',
    'TIGI Bed Head', 'Viktor & Rolf', 'bareMinerals',
    'Beauty Works', 'Calvin Klein', 'Chantecaille',
    'Ciate London', 'Comfort Zone', 'Dr. Hauschka',
    'Estée Lauder', 'Hanz de Fuko', 'Hello Sunday',
    'Invisibobble', 'Issey Miyake', 'IT Cosmetics',
    'Kevyn Aucoin', 'Leonor Greyl', 'Living Proof',
    'Mermade Hair', 'Michael Kors', 'Mio Skincare',
    'Molton Brown', 'Natura Bissé', 'OPI Nailcare',
    'Perricone MD', 'Prada Beauty', 'Ralph Lauren',
    'RoC Skincare', 'Sally Hansen', 'Serge Lutens',
    'Shark Beauty', 'SILKE London', 'skinChemists',
    'Skin Doctors', 'Sleek MakeUP', 'The Ordinary',
    'Bobbi Brown', 'Bondi Boost', 'Bondi Sands',
    "Burt's Bees", 'Dear Klairs', 'Dermalogica',
    'Doll Beauty', 'Dr. Loretta', 'Embryolisse',
    'Emma Hardie', 'KIKO MILANO', 'Marc Jacobs',
    'Moroccanoil', 'PIXI Beauty', 'PRAI Beauty',
    'Riemann P20', 'The Nue Co.', 'Urban Decay',
    'Virtue Labs', 'AMELIORATE', 'Balance Me',
    'CLOUD NINE', 'Coco & Eve', 'Dr. Brandt',
    'Dr. PAWPAW', 'Frank Body', 'Illamasqua',
    'Jack Black', 'Jimmy Choo', 'KVD Beauty',
    "L'OCCITANE", 'Lime Crime', 'Loving Tan',
    'Manucurist', 'Max Factor', 'Maybelline',
    'MenScience', 'Myvitamins', 'Neutrogena',
    'Omorovicza', 'Patchology', 'Philosophy',
    'RevitaLash', 'Sand & Sky', 'Skinny Tan',
    'Skinstitut', 'St. Tropez', 'StriVectin',
    'Supergoop!', 'Tanologist', 'This Works',
    'Toni & Guy', 'Tweezerman', 'Antipodes',
    'Bali Body', 'BIOEFFECT', 'Biossance',
    'Collistar', 'Color Wow', 'Curlsmith',
    'Easilocks', 'Elie Saab', 'Face Halo',
    'Fillerina', 'Hairburst', 'Hair Gain',
    'Herbivore', 'Hugo Boss', 'Kérastase',
    'Lancaster', 'Lixirskin', 'Liz Earle',
    'Montblanc', 'Nailberry', 'Nails.Inc',
    'Neostrata', 'Nip + Fab', 'Pureology',
    'RapidLash', 'Sachajuan', 'St. Moriz',
    'Too Faced', 'TriPollar', 'Valentino',
    'Vera Wang', 'Vida Glow', 'ZitSticka',
    'Acnecide', 'Algenist', 'BaByliss', 'Bioderma',
    'BioIonic', 'Biotherm', 'Bouclème', 'Burberry',
    'By Terry', 'Caudalie', 'Cetaphil', 'Clinique',
    'Dr.Jart+', 'Dr. Levy', 'Dr. Lipp', 'EcoTools',
    'Erborian', 'Fade Out', 'Flawless', 'Gallinée',
    'Gatineau', 'Giovanni', 'GIVENCHY', 'Glow Hub',
    'Goldwell', 'GUERLAIN', 'HoMedics', 'Intimina',
    'KeraCare', 'Kerasilk', 'Lanolips', 'Mama Mio',
    'Moschino', 'Mr Blanc', 'Nourkrin', 'NUDESTIX',
    'Philip B', 'Piz Buin', 'Sambucol', 'Shiseido',
    'STARSKIN', 'Tan Luxe', 'TOM FORD', 'Ultrasun',
    'UpCircle', 'Viviscal', 'Waterpik', 'WetBrush',
    '111SKIN', 'Alpha H', 'As I Am', 'Balmain',
    'Batiste', 'benefit', 'Biolage', 'Biretix',
    'Bulldog', 'BVLGARI', 'Clairol', 'Cowshed',
    'Decorté', 'delilah', 'Dunhill', 'Eucerin',
    'Eve Lom', 'FaceGym', 'Filorga', 'Garnier',
    'Hurraw!', 'Imedeen', 'Klorane', 'Lacoste',
    'Lancôme', 'LANEIGE', 'MineTan', 'Moncler',
    'MZ Skin', 'Nail HQ', 'Nanogen', 'Noughty',
    'OLAPLEX', 'Origins', 'Pattern', 'Rabanne',
    'Regaine', 'Rituals', 'Sensica', 'Sentier',
    'StylPro', 'Sun Bum', 'Surratt', 'Thayers',
    'Trilogy', 'Viseart', 'Westlab', 'ARKIVE',
    'Aveeno', 'Azzaro', 'CeraVe', 'Cult51',
    'Denman', 'Diesel', 'DIRTEA', 'Drybar',
    'ELEMIS', 'Eylure', 'Goutal', 'HERMÈS',
    'Inglot', 'Kitsch', 'KORRES', "L'Anza",
    'La Mer', 'LIERAC', 'Lumene', 'Marvis',
    'Matrix', 'Mavala', 'Medik8', 'Morphe',
    'MUGLER', 'NIOXIN', 'NuFACE', 'Oral-B',
    'Ouidad', 'Parlux', 'Polaar', 'PURITO',
    'Redken', 'ReVive', 'Revlon', 'Rimmel',
    'Rodial', 'Sensse', "Silk'n", 'Simple',
    'So Eco', 'Uriage', 'Weleda', 'Yes To',
    'Zelens', 'AERIN', 'Aesop', 'AHAVA',
    'Aveda', 'Avène', 'Braun', 'Cantu',
    'Chloé', 'COACH', 'COOLA', 'CosRX',
    'Curél', 'Cutex', 'DuWop', 'Essie',
    'Eyeko', 'Faace', 'FOREO', 'Fresh',
    'INIKA', 'JASON', 'Joico', 'Luxie',
    'Mauli', 'men-ü', 'Murad', 'Mylee',
    'Natio', 'Oh K!', 'OSKIA', 'PAYOT',
    'Phyto', 'Rahua', 'Sigma', 'Stila',
    'Sukin', 'Sweed', 'Tocca', 'VICHY',
    'Zoeva', 'bybi', 'DKNY', 'ESPA',
    'GLOV', 'KISS', 'LELO', 'Lynx',
    'NARS', 'NUXE', 'OUAI', 'Slip',
    'TIGI', 'Tria', 'utan', 'VOYA',
    'VUSH', 'WAHL', 'Wild', 'CND',
    'DHC', 'Duo', 'GHD', 'KMS',
    'OGX', 'ORS', 'OTO', 'Pai',
    'Rio', 'SVR', 'T3',
]

live_brands = fetch_brand_list()
brand_list = sorted(live_brands or CACHED_BRAND_LIST, key=len, reverse=True)
print(f"Brand list contains {len(brand_list)} brands ({'live' if live_brands else 'cached'})")

Brand list contains 396 brands (cached)


In [ ]:
# Extract brand from product name

def normalize_text(text):
    return re.sub(r'\W+', ' ', text).strip().lower()


def match_brand(product_name, brand_list):
    """Return the best matching brand from brand_list for a given product name."""
    norm_name = normalize_text(product_name)
    for brand in brand_list:
        if normalize_text(brand) in norm_name:
            return brand.title()
    return None


# Priority: JSON-LD brand from product page → name-based match
sc_data['brand'] = sc_data['brand_from_page'].where(
    sc_data['brand_from_page'].notna(),
    sc_data['product_name'].apply(lambda x: match_brand(x, brand_list))
)

In [ ]:
# Manual brand fixes for edge cases not caught above

MANUAL_BRANDS = [
    'Avene', 'Erno Laszlo', 'NIOD', 'DECLÉOR', 'Darphin', 'GLAMGLOW',
    'Freezeframe', 'MONU', 'Jurlique', 'Salcura', 'Benton', 'FARMACY',
    'Instant Effects', 'RapidEye', 'MÁDARA', 'Bubble T', 'Sea Magik',
    'ManCave', 'Love Boo',
]


def fill_missing_brand(product_name):
    for brand in MANUAL_BRANDS:
        if brand.lower() in product_name.lower():
            return brand.title()
    return None


mask = sc_data['brand'].isnull()
sc_data.loc[mask, 'brand'] = sc_data.loc[mask, 'product_name'].apply(fill_missing_brand)

# Normalise known spelling variants
sc_data['brand'] = sc_data['brand'].replace({
    'Avene': 'Avène',
    'Bloom And Blossom': 'Bloom & Blossom',
    "L'Oréal Men Expert": "L'Oréal Paris",
    "L\u2019Oréal Paris": "L'Oréal Paris",
    "L\u2019Oréal Professionnel": "L'Oréal Paris",
})

print(f"Null brands remaining: {sc_data['brand'].isnull().sum()}")
sc_data.head()

Null brands remaining: 170


,product_name,product_url,product_type,ingredients,price,updated_price,product_rating,product_image_url,available,brand_from_page,price_change,brand
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",5.2,6.10,4.4096,https://main.thgimages.com/?url=https://static...,True,NaN,0.90,The Ordinary
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",13.0,16.50,4.2962,https://main.thgimages.com/?url=https://static...,True,NaN,3.50,Cerave
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,"['sodium hyaluronate', 'sodium hyaluronate', '...",6.2,34.63,4.4748,https://main.thgimages.com/?url=https://static...,True,NaN,28.43,The Ordinary
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,Moisturiser,"['ammonium lactate', 'c12-15', 'glycerin', 'pr...",22.5,30.60,4.4754,https://main.thgimages.com/?url=https://static...,True,NaN,8.10,Ameliorate
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,Moisturiser,"['glycerin', 'cetearyl alcohol', 'capric trigl...",16.0,15.50,4.7049,https://main.thgimages.com/?url=https://static...,True,NaN,-0.50,Cerave


In [ ]:
# Apply corrected brands dictionary

corrected_brands_df = pd.read_csv('corrected_brands_dictionary.csv', index_col=0)
merged_df = sc_data.merge(corrected_brands_df, on='product_name', how='left', suffixes=('', '_corrected'))
sc_data['brand'] = sc_data['brand'].combine_first(merged_df['brand_corrected'])

print(f"Null brands after correction: {sc_data['brand'].isnull().sum()}")
sc_data.head()

Null brands after correction: 66


,product_name,product_url,product_type,ingredients,price,updated_price,product_rating,product_image_url,available,brand_from_page,price_change,brand
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",5.2,6.10,4.4096,https://main.thgimages.com/?url=https://static...,True,NaN,0.90,The Ordinary
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",13.0,16.50,4.2962,https://main.thgimages.com/?url=https://static...,True,NaN,3.50,Cerave
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,"['sodium hyaluronate', 'sodium hyaluronate', '...",6.2,34.63,4.4748,https://main.thgimages.com/?url=https://static...,True,NaN,28.43,The Ordinary
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,Moisturiser,"['ammonium lactate', 'c12-15', 'glycerin', 'pr...",22.5,30.60,4.4754,https://main.thgimages.com/?url=https://static...,True,NaN,8.10,Ameliorate
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,Moisturiser,"['glycerin', 'cetearyl alcohol', 'capric trigl...",16.0,15.50,4.7049,https://main.thgimages.com/?url=https://static...,True,NaN,-0.50,Cerave


In [ ]:
# Summary

total     = len(sc_data)
available = int(sc_data['available'].sum())
null_price = sc_data['updated_price'].isnull().sum()
null_rating= sc_data['product_rating'].isnull().sum()
null_brand = sc_data['brand'].isnull().sum()

print(f"Total products : {total}")
print(f"Available      : {available}  |  Discontinued: {total - available}")
print(f"Missing price  : {null_price}")
print(f"Missing rating : {null_rating}")
print(f"Missing brand  : {null_brand}")
sc_data.head()

Total products : 1138
Available      : 1032  |  Discontinued: 106
Missing price  : 192
Missing rating : 222
Missing brand  : 66


,product_name,product_url,product_type,ingredients,price,updated_price,product_rating,product_image_url,available,brand_from_page,price_change,brand
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",5.2,6.10,4.4096,https://main.thgimages.com/?url=https://static...,True,NaN,0.90,The Ordinary
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",13.0,16.50,4.2962,https://main.thgimages.com/?url=https://static...,True,NaN,3.50,Cerave
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,"['sodium hyaluronate', 'sodium hyaluronate', '...",6.2,34.63,4.4748,https://main.thgimages.com/?url=https://static...,True,NaN,28.43,The Ordinary
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,Moisturiser,"['ammonium lactate', 'c12-15', 'glycerin', 'pr...",22.5,30.60,4.4754,https://main.thgimages.com/?url=https://static...,True,NaN,8.10,Ameliorate
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,Moisturiser,"['glycerin', 'cetearyl alcohol', 'capric trigl...",16.0,15.50,4.7049,https://main.thgimages.com/?url=https://static...,True,NaN,-0.50,Cerave


In [ ]:
# Save refreshed dataset

output_cols = [
    'product_name', 'product_url', 'product_type', 'ingredients',
    'price', 'updated_price', 'price_change',
    'product_rating', 'product_image_url', 'brand', 'available',
]

out_df = sc_data[output_cols].copy()
out_df.to_csv('skincare_products_2026.csv', index=False)
print(f"Saved {len(out_df)} rows to skincare_products_2026.csv")

Saved 1138 rows to skincare_products_2026.csv


In [ ]:
# Quick sanity check on saved file

df = pd.read_csv('skincare_products_2026.csv')
print(df.shape)
df.head()

(1138, 11)


,product_name,product_url,product_type,ingredients,price,updated_price,price_change,product_rating,product_image_url,brand,available
0,The Ordinary Natural Moisturising Factors + HA...,https://www.lookfantastic.com/the-ordinary-nat...,Moisturiser,"['capric triglyceride', 'cetyl alcohol', 'prop...",5.2,6.10,0.90,4.4096,https://main.thgimages.com/?url=https://static...,The Ordinary,True
1,CeraVe Facial Moisturising Lotion SPF 25 52ml,https://www.lookfantastic.com/cerave-facial-mo...,Moisturiser,"['homosalate', 'glycerin', 'octocrylene', 'eth...",13.0,16.50,3.50,4.2962,https://main.thgimages.com/?url=https://static...,Cerave,True
2,The Ordinary Hyaluronic Acid 2% + B5 Hydration...,https://www.lookfantastic.com/the-ordinary-hya...,Moisturiser,"['sodium hyaluronate', 'sodium hyaluronate', '...",6.2,34.63,28.43,4.4748,https://main.thgimages.com/?url=https://static...,The Ordinary,True
3,AMELIORATE Transforming Body Lotion 200ml,https://www.lookfantastic.com/ameliorate-trans...,Moisturiser,"['ammonium lactate', 'c12-15', 'glycerin', 'pr...",22.5,30.60,8.10,4.4754,https://main.thgimages.com/?url=https://static...,Ameliorate,True
4,CeraVe Moisturising Cream 454g,https://www.lookfantastic.com/cerave-moisturis...,Moisturiser,"['glycerin', 'cetearyl alcohol', 'capric trigl...",16.0,15.50,-0.50,4.7049,https://main.thgimages.com/?url=https://static...,Cerave,True
